# PYTHON-12. Продвинутые методы работы с данными в Pandas (Урок 4)
## Практическая работа на данных Krisha.kz: 1-комнатные квартиры в Астане (районы Есиль и Нура)

**Критерии выборки:**
- **Город:** Астана
- **Районы:** Есильский р-н и Нура р-н
- **Комнатность:** строго 1-комнатные (`room_count == 1`)
- **Год постройки:** от 2016 года (`construction_year >= 2016`)
- **Период объявлений:** с января 2026 года (`date >= '2026-01-01'`)
- **Источник:** `krisha_astana_esil_nura_2026.csv` (750 объявлений, собранных парсером Krisha.kz)


## 1. Введение и подготовка типов данных
*(Аналог Заданий 1–2 из `data_manipulation_1.ipynb`)*

Загрузим датасет, проверим соответствие критериям и восстановим типы `datetime` и `category` (так как CSV не сохраняет типы столбцов).


In [1]:
import re
import pandas as pd
import numpy as np

krisha_data = pd.read_csv('krisha_astana_esil_nura_2026.csv')
krisha_df = krisha_data.copy()

# Проверка критериев выборки
krisha_df = krisha_df[
    (krisha_df['room_count'] == 1) &
    (krisha_df['district'].isin(['Есильский р-н', 'Нура р-н'])) &
    (krisha_df['construction_year'] >= 2016) &
    (krisha_df['date'] >= '2026-01-01')
].copy()

# Базовые признаки для анализа
krisha_df['Price_m2'] = (krisha_df['price'] / krisha_df['area']).round().astype(int)
krisha_df['FloorType'] = np.where(
    krisha_df['floor'] == 1, 'первый',
    np.where(krisha_df['floor'] == krisha_df['floor_count'], 'последний', 'средний')
)
print(f'Размер выборки: {krisha_df.shape[0]} строк, {krisha_df.shape[1]} столбцов')
krisha_df.head(3)


Размер выборки: 750 строк, 21 столбцов


,city,name,district,address,price,price_text,price_is_from,owner,complex_name,house_type,...,ceiling_height,bathroom_info,area,room_count,floor,floor_count,url,date,Price_m2,FloorType
0,Астана,1-комнатная квартира · 42.78 м² · 8/21 этаж,Нура р-н,"Нура р-н, Бухар жырау 20",41838840,"41,838,840 ₸",True,Новостройка ЖК «Qasteev» от застройщика «BI Gr...,Qasteev,NaN,...,3 м,совмещенный,42.78,1,8,21,https://krisha.kz/a/show/1010216007,2026-11-23,978000,средний
1,Астана,1-комнатная квартира · 37.43 м² · 6/14 этаж,Есильский р-н,"Есильский р-н, Е 915 11 — Улы дала",26000000,"26,000,000 ₸",False,Крыша Агент,Jetisu Kerbez Comfort,монолитный,...,3 м,совмещенный,37.43,1,6,14,https://krisha.kz/a/show/1017116184,2026-11-23,694630,средний
2,Астана,1-комнатная квартира · 38 м² · 3/9 этаж,Есильский р-н,"Есильский р-н, мкр Комсомольский, Улы Дала Е 908",19500000,"19,500,000 ₸",False,Хозяин недвижимости,Uly Dala Park,монолитный,...,3 м,раздельный,38.00,1,3,9,https://krisha.kz/a/show/1017150440,2026-11-23,513158,средний


### Задание 1. Преобразование `date` в `datetime` и анализ по кварталам
> Преобразуйте столбец `date` в формат `datetime`, выделите квартал (`quarter`) публикации объявления, найдите второй по популярности квартал и количество объявлений в нём.


In [2]:
krisha_df['date'] = pd.to_datetime(krisha_df['date'])
krisha_df['Quarter'] = krisha_df['date'].dt.quarter

quarter_counts = krisha_df['Quarter'].value_counts()
print(quarter_counts)
print(f'Второй по популярности квартал: {quarter_counts.index[1]}-й квартал ({quarter_counts.iloc[1]} объявлений)')


Quarter
2    226
1    199
3    190
4    135
Name: count, dtype: int64
Второй по популярности квартал: 1-й квартал (199 объявлений)


**Результат по данным:**
- 2-й квартал: 226 объявлений
- **1-й квартал (второй по популярности): 199 объявлений**
- 3-й квартал: 190 объявлений
- 4-й квартал: 135 объявлений


### Задание 2. Преобразование категориальных признаков в `category`
> Преобразуйте все столбцы, в которых меньше 150 уникальных значений, в тип `category`, исключив числовые/служебные поля (`date`, `room_count`, `floor`, `floor_count`, `construction_year`).


In [3]:
exclude_list = ['date', 'room_count', 'floor', 'floor_count', 'construction_year']
mem_before = krisha_df.memory_usage(deep=True).sum() / (1024 ** 2)

for col in krisha_df.columns:
    if col not in exclude_list and krisha_df[col].nunique() < 150:
        krisha_df[col] = krisha_df[col].astype('category')

mem_after = krisha_df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f'Память до: {mem_before:.3f} МБ -> после: {mem_after:.3f} МБ (экономия {(1 - mem_after/mem_before)*100:.1f}%)')


Память до: 0.835 МБ -> после: 0.484 МБ (экономия 42.0%)


## 2. Сортировка данных в DataFrame (`sort_values`)
*(Аналог раздела 2 и Заданий 3–4 из `data_manipulation_1.ipynb`)*


In [4]:
# Сортировка по одному столбцу (топ-5 самых дорогих по цене за м²)
krisha_df.sort_values(by='Price_m2', ascending=False)[
    ['district', 'complex_name', 'area', 'price', 'Price_m2', 'construction_year']
].head(5)


,district,complex_name,area,price,Price_m2,construction_year
0,Нура р-н,Qasteev,42.78,41838840,978000,2027
409,Есильский р-н,Only Sun,37.40,36400000,973262,2026
202,Есильский р-н,Akbulak Riviera,41.40,40100000,968599,2026
102,Есильский р-н,Only Sun,36.80,34700000,942935,2024
525,Есильский р-н,Nova City,36.70,34600000,942779,2025


In [5]:
# Сортировка по нескольким столбцам: сначала район (по возр.), затем цена за м² (по убыв.)
krisha_df.sort_values(by=['district', 'Price_m2'], ascending=[True, False])[
    ['district', 'complex_name', 'area', 'price', 'Price_m2']
].head(10)


,district,complex_name,area,price,Price_m2
409,Есильский р-н,Only Sun,37.4,36400000,973262
202,Есильский р-н,Akbulak Riviera,41.4,40100000,968599
102,Есильский р-н,Only Sun,36.8,34700000,942935
525,Есильский р-н,Nova City,36.7,34600000,942779
372,Есильский р-н,Uly Dala Park,43.6,40700000,933486
42,Есильский р-н,Viva Plaza,40.3,37600000,933002
483,Есильский р-н,Uly Dala Park,37.7,35100000,931034
635,Есильский р-н,Croco,39.7,36700000,924433
320,Есильский р-н,Croco,39.6,36600000,924242
137,Есильский р-н,Croco,49.2,45400000,922764


### Задания 3 и 4 (по методике урока 4)
1. Отсортируйте таблицу по `Price_m2` по убыванию, обновите индексы (`ignore_index=True`). Какая площадь (`area`) и цена (`price`) в строке с индексом `18`?
2. Найдите квартиры площадью `area > 40` м², отсортируйте по возрастанию `construction_year` и по убыванию `Price_m2`, обновите индексы. Какая цена будет у объекта в строке `18`?


In [6]:
# Задание 3
sorted_pm2 = krisha_df.sort_values(by='Price_m2', ascending=False, ignore_index=True)
row_18_task3 = sorted_pm2.loc[18, ['district', 'complex_name', 'area', 'price', 'Price_m2']]
print('Задание 3 (строка 18):\n', row_18_task3)

# Задание 4
filtered_40 = krisha_df[krisha_df['area'] > 40].sort_values(
    by=['construction_year', 'Price_m2'],
    ascending=[True, False],
    ignore_index=True
)
row_18_task4 = filtered_40.loc[18, ['district', 'complex_name', 'construction_year', 'area', 'price', 'Price_m2']]
print('\nЗадание 4 (строка 18):\n', row_18_task4)


Задание 3 (строка 18):
 district        Есильский р-н
complex_name     Sensata City
area                     35.6
price                32400000
Price_m2               910112
Name: 18, dtype: object

Задание 4 (строка 18):
 district             Есильский р-н
complex_name            Expo Plaza
construction_year             2017
area                          43.1
price                     33500000
Price_m2                    777262
Name: 18, dtype: object


**Проверенные ответы:**
- **Задание 3 (строка 18):** `area = 35.6` м², `price = 32 400 000 ₸`, `Price_m2 = 910 112 ₸/м²`.
- **Задание 4 (строка 18 из 454 объектов с площадью > 40 м²):** `price = 33 500 000 ₸`, `construction_year = 2017`, `Есильский р-н`, ЖК `Expo Plaza`.


## 3. Группировка данных (`groupby` и `agg`)
*(Аналог раздела 3 из `data_manipulation_1.ipynb`)*


In [7]:
# Сравнение районов: количество, среднее, медиана, мин и макс
district_stats = krisha_df.groupby('district', observed=True).agg(
    listings=('price', 'count'),
    mean_price=('price', 'mean'),
    median_price=('price', 'median'),
    mean_pm2=('Price_m2', 'mean'),
    median_pm2=('Price_m2', 'median'),
    mean_area=('area', 'mean')
).round(0)
district_stats


,listings,mean_price,median_price,mean_pm2,median_pm2,mean_area
district,,,,,,
Есильский р-н,373,32213906.0,32400000.0,766647.0,764434.0,42.0
Нура р-н,377,28175348.0,28100000.0,678902.0,681564.0,42.0


In [8]:
# Группировка по типу дома и району с несколькими агрегациями
krisha_df.groupby(['district', 'house_type'], observed=True)['Price_m2'].agg(
    ['count', 'mean', 'median', 'min', 'max']
).round(0)


count      mean    median     min     max
district      house_type                                           
Есильский р-н кирпичный      66  805385.0  818724.0  563246  968599
              монолитный    304  759056.0  757375.0  513158  973262
Нура р-н      кирпичный      42  712639.0  710422.0  572973  864919
              монолитный    309  679921.0  683453.0  494845  853881
              панельный      22  594631.0  588516.0  508621  762402

**Выводы по группировке:**
- **Есильский р-н (373 объявления):** средняя цена `32 213 906 ₸` (медиана `32 400 000 ₸`), средняя цена за м² `766 647 ₸/м²`.
- **Нура р-н (377 объявлений):** средняя цена `28 175 348 ₸` (медиана `28 100 000 ₸`), средняя цена за м² `678 902 ₸/м²`.
- **По типу дома:** кирпичные дома самые дорогие (`818 724 ₸/м²` медиана в Есиле, `710 422 ₸/м²` в Нуре), панельные встречаются только в Нуре (`588 516 ₸/м²` медиана).


## 4. Сводные таблицы (`unstack`, `pivot_table`, `MultiIndex`)
*(Аналог раздела 4 из `data_manipulation_1.ipynb`)*


In [9]:
# 1) Сводная таблица через groupby + unstack: медианная цена за м² по району и типу дома
krisha_df.groupby(['district', 'house_type'], observed=True)['Price_m2'].median().unstack().round(0)


house_type,кирпичный,монолитный,панельный
district,,,
Есильский р-н,818724.0,757375.0,NaN
Нура р-н,710422.0,683453.0,588516.0


In [10]:
# 2) Сводная таблица через pivot_table: медианная площадь (area) по типу дома и категории этажа
pivot_area = krisha_df.pivot_table(
    values='area',
    index='FloorType',
    columns='house_type',
    aggfunc='median',
    observed=True
).round(2)
pivot_area


house_type,кирпичный,монолитный,панельный
FloorType,,,
первый,41.90,38.5,42.0
последний,42.45,41.3,37.2
средний,43.50,41.7,40.3


In [11]:
# 3) Многомерная сводная таблица (MultiIndex): медианная и средняя цена по району, типу этажа и типу дома
pivot_multi = krisha_df.pivot_table(
    values='Price_m2',
    index=['district', 'FloorType'],
    columns='house_type',
    aggfunc=['median', 'count'],
    observed=True
).round(0)
pivot_multi


median                          count             \
house_type              кирпичный монолитный панельный кирпичный монолитный   
district      FloorType                                                       
Есильский р-н первый     714706.0   738041.0       NaN       5.0       23.0   
              последний  778947.0   757692.0       NaN       9.0       19.0   
              средний    835556.0   758518.0       NaN      52.0      262.0   
Нура р-н      первый     727858.0   657470.0  614286.0       2.0       20.0   
              последний  575117.0   673626.0  591398.0       3.0       26.0   
              средний    720000.0   690418.0  585588.0      37.0      263.0   

                                   
house_type              панельный  
district      FloorType            
Есильский р-н первый          NaN  
              последний       NaN  
              средний         NaN  
Нура р-н      первый          1.0  
              последний       1.0  
              средний        20.0

In [12]:
# 4) Задание раздела 4: медианная цена (price) по продавцу (owner) и типу дома (house_type)
# Найти продавца с максимальной медианной ценой для монолитных домов (среди популярных продавцов)
top_owners = krisha_df['owner'].value_counts().nlargest(8).index
pivot_owner = krisha_df[krisha_df['owner'].isin(top_owners)].pivot_table(
    values='price',
    index='owner',
    columns='house_type',
    aggfunc='median',
    observed=True
).round(0)
print(pivot_owner)
print('\nМаксимум для монолитных домов:', pivot_owner['монолитный'].idxmax(), pivot_owner['монолитный'].max())


house_type             кирпичный  монолитный   панельный
owner                                                   
BI Group Риелти       32550000.0  30000000.0  24550000.0
Golden Key            31200000.0  31000000.0  30650000.0
Sensata Real Estate   31300000.0  28300000.0  23550000.0
Витрина недвижимости  33050000.0  30350000.0  23150000.0
Крыша Агент           33650000.0  28900000.0  23750000.0
Специалист            32500000.0  30200000.0  24850000.0
Хозяин недвижимости   31400000.0  29400000.0  23400000.0
Этажи Астана          34900000.0  29650000.0  33000000.0

Максимум для монолитных домов: Golden Key 31000000.0


## 5. Объединение нескольких таблиц (`concat`, `join`, `merge`)
*(Аналог разделов 5–7 из `data_manipulation_2.ipynb`)*

На практике выгрузки с Krisha.kz часто собираются по отдельным районам или дополняются справочниками ЖК. Смоделируем этот сценарий на наших данных:
1. Разделим данные на два DataFrame (`esil_df` и `nura_df`) и объединим их обратно через `pd.concat()`.
2. Создадим справочник ЖК (`complex_ref`) и присоединим его через `merge()` и `join()`.


In [13]:
# 1. pd.concat: объединение выгрузок двух районов
esil_df = krisha_df[krisha_df['district'] == 'Есильский р-н'].copy()
nura_df = krisha_df[krisha_df['district'] == 'Нура р-н'].copy()

combined_df = pd.concat([esil_df, nura_df], ignore_index=True)
print(f'Есиль: {len(esil_df)} + Нура: {len(nura_df)} -> concat: {len(combined_df)} строк')


Есиль: 373 + Нура: 377 -> concat: 750 строк


In [14]:
# 2. merge(): обогащение объявлений статистикой по ЖК
complex_ref = (
    krisha_df.groupby('complex_name', observed=True)
    .agg(complex_listings=('price', 'count'), complex_mean_pm2=('Price_m2', 'mean'))
    .round(0)
    .reset_index()
)

merged_df = krisha_df.merge(complex_ref, on='complex_name', how='left')
merged_df['diff_from_complex_pm2'] = merged_df['Price_m2'] - merged_df['complex_mean_pm2']
print(f'Строк до merge: {len(krisha_df)}, после merge: {len(merged_df)}')
merged_df[['district', 'complex_name', 'Price_m2', 'complex_mean_pm2', 'diff_from_complex_pm2']].head(5)


Строк до merge: 750, после merge: 750


,district,complex_name,Price_m2,complex_mean_pm2,diff_from_complex_pm2
0,Нура р-н,Qasteev,978000,783081.0,194919.0
1,Есильский р-н,Jetisu Kerbez Comfort,694630,694630.0,0.0
2,Есильский р-н,Uly Dala Park,513158,794902.0,-281744.0
3,Есильский р-н,Impulse.Sport,609000,749626.0,-140626.0
4,Нура р-н,ABAI JOLY AVENUE,612557,612557.0,0.0


In [15]:
# 3. join(): объединение по индексу (справочник характеристик районов)
district_info = pd.DataFrame({
    'zone': ['Левый берег (центр/Экспо)', 'Левый берег (запад/Туран)'],
    'district_mean_pm2': [766647, 678902]
}, index=['Есильский р-н', 'Нура р-н'])

joined_df = krisha_df.set_index('district').join(district_info, how='left').reset_index()
joined_df[['district', 'zone', 'complex_name', 'Price_m2', 'district_mean_pm2']].head(5)


,district,zone,complex_name,Price_m2,district_mean_pm2
0,Нура р-н,Левый берег (запад/Туран),Qasteev,978000,678902
1,Есильский р-н,Левый берег (центр/Экспо),Jetisu Kerbez Comfort,694630,766647
2,Есильский р-н,Левый берег (центр/Экспо),Uly Dala Park,513158,766647
3,Есильский р-н,Левый берег (центр/Экспо),Impulse.Sport,609000,766647
4,Нура р-н,Левый берег (запад/Туран),ABAI JOLY AVENUE,612557,678902


## 6. Закрепление знаний (итоговые задачи)
*(Аналог раздела 8 из `data_manipulation_2.ipynb`)*

1. Используя регулярное выражение (`re`), извлеките этажность из текстового поля `name` и проверьте совпадение с `floor`/`floor_count`.
2. Найдите ЖК (с количеством объявлений > 10), которые имеют **наибольшую** и **наименьшую** среднюю цену за м².
3. Постройте сводную таблицу зависимости средней цены за м² от года постройки (`construction_year`) и района (`district`).


In [16]:
# 1. Извлечение этажа и этажности из заголовка name с помощью re
def extract_floor_pair(text):
    match = re.findall(r'(\d+)/(\d+)\s*этаж', str(text))
    return (int(match[0][0]), int(match[0][1])) if match else (None, None)

extracted = merged_df['name'].apply(extract_floor_pair)
print('Совпадение floor с заголовком:', (extracted.str[0] == merged_df['floor']).mean() * 100, '%')

# 2. Топ и анти-топ ЖК среди тех, где > 10 объявлений
popular_complexes = (
    merged_df.groupby(['complex_name', 'district'], observed=True)
    .agg(listings=('price', 'count'), mean_pm2=('Price_m2', 'mean'), mean_price=('price', 'mean'))
    .query('listings > 10')
    .sort_values('mean_pm2', ascending=False)
    .round(0)
)
print('\nТоп-3 ЖК (>10 объявлений):\n', popular_complexes.head(3))
print('\nСамые доступные ЖК (>10 объявлений):\n', popular_complexes.tail(3))

# 3. Сводная таблица: средняя цена за м² по году постройки и району
year_district_pivot = merged_df.pivot_table(
    values='Price_m2',
    index='construction_year',
    columns='district',
    aggfunc='mean',
    observed=True
).round(0)
year_district_pivot


Совпадение floor с заголовком: 99.46666666666667 %

Топ-3 ЖК (>10 объявлений):
                                listings  mean_pm2  mean_price
complex_name    district                                     
Croco           Есильский р-н        20  813939.0  34710000.0
Nova City       Есильский р-н        26  797927.0  33569231.0
Akbulak Riviera Есильский р-н        24  796127.0  34729167.0

Самые доступные ЖК (>10 объявлений):
                         listings  mean_pm2  mean_price
complex_name  district                                
Turan Tower   Нура р-н        29  664577.0  26493103.0
MoD. Standard Нура р-н        28  664131.0  28167857.0
Sezim Qala    Нура р-н        22  594631.0  24954545.0


district,Есильский р-н,Нура р-н
construction_year,,
2016,690220.0,584043.0
2017,700620.0,620135.0
2018,706947.0,624057.0
2019,700929.0,616408.0
2020,740825.0,647390.0
2021,732633.0,653787.0
2022,771866.0,692628.0
2023,789135.0,700846.0
2024,801147.0,706113.0
